# ECG Language Processing

## Libraries and Packages

In [1]:
!pip install neurokit2
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 708.4/708.4 kB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 129.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [neurokit2]/2 [neurokit2]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 117.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [wfdb]2m 8/10 [aiohttp]


In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cu124
True


In [5]:
import boto3
import io
import os
import gc

import pickle
import neurokit2 as nk
import pandas as pd
import numpy as np

from tqdm import tqdm
import time
import random
from random import shuffle, randint, randrange
import matplotlib.pyplot as plt

import math
import scipy
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.functional import relu
import torch.nn.functional as F

import wfdb

import warnings
warnings.simplefilter("ignore")

In [6]:
ptb_cs = 'mimic'

In [7]:



if ptb_cs == 'mimic':
    print('mimic')
    suff = './pickles/MIMIC/'
os.makedirs(suff, exist_ok=True)

mimic


# Functions to access the Data and create files_names

### (skipped) PTB

In [10]:
if ptb_cs == 'ptb':
    s3_client = boto3.client('s3')

    bucket_name = 'your_s3'
    prefix = 'ptbxl/physionet.org/files/challenge-2021/1.0.3/training/ptb-xl/'

    def list_all_folders(bucket, prefix, delimiter='/'):
        folders = []
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter=delimiter):
            folders.extend([cp['Prefix'].rstrip('/').split('/')[-1] for cp in page.get('CommonPrefixes', [])])
        return folders

    def read_data(folder_name, file_name):
        prefix = 'ptbxl/physionet.org/files/challenge-2021/1.0.3/training/ptb-xl/'
        data_key = f'{prefix}{folder_name}/{file_name}'
        response = s3_client.get_object(Bucket=bucket_name, Key=data_key)
        mat_file = response['Body'].read()
        mat_data = scipy.io.loadmat(io.BytesIO(mat_file))

        lead = 1
        df_main = pd.DataFrame({k: mat_data[k][lead] if isinstance(mat_data[k], np.ndarray) else mat_data[k] for k in mat_data.keys()})
        ecg_signal = df_main.iloc[:, 0].values
        return ecg_signal

    folder_names = list_all_folders(bucket_name, prefix)
    sampling_rate = 500

    print('ptb')


    s3_client = boto3.client('s3')

    bucket_name = 'your_s3'
    base_prefix = 'ptbxl/physionet.org/files/challenge-2021/1.0.3/training/ptb-xl/'

    folder_names_g = []

    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket_name, Prefix=base_prefix, Delimiter='/'):
        if 'CommonPrefixes' in page:
            folder_names_g.extend([prefix['Prefix'].split('/')[-2] for prefix in page['CommonPrefixes']])

    print("Found folders:", folder_names_g)

    files_names = []

    for folder_name in folder_names_g:
        prefix = f"{base_prefix}{folder_name}/"

        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                if obj['Key'].endswith('.mat'):
                    file_name = obj['Key'].split('/')[-1]
                    files_names.append((folder_name, file_name))

    print("Sample files for read_data compatibility:")
    for folder_name, file_name in files_names[:10]:
        print(f"Folder: {folder_name}, File: {file_name}")

    print(f"Total files saved: {len(files_names)}")

    folder_names = files_names
    print(folder_names[0])
    print(len(folder_names))

ptb
Found folders: ['g1', 'g10', 'g11', 'g12', 'g13', 'g14', 'g15', 'g16', 'g17', 'g18', 'g19', 'g2', 'g20', 'g21', 'g22', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9']
Sample files for read_data compatibility:
Folder: g1, File: HR00001.mat
Folder: g1, File: HR00002.mat
Folder: g1, File: HR00003.mat
Folder: g1, File: HR00004.mat
Folder: g1, File: HR00005.mat
Folder: g1, File: HR00006.mat
Folder: g1, File: HR00007.mat
Folder: g1, File: HR00008.mat
Folder: g1, File: HR00009.mat
Folder: g1, File: HR00010.mat
Total files saved: 21837
('g1', 'HR00001.mat')
21837


#### CS

In [ ]:
if ptb_cs == 'cs':
    s3_client = boto3.client('s3')

    bucket_name = 'your_s3'
    prefix = 'Chapman_Shaoxing/ECGDataDenoised/'

    def list_all_folders(bucket_name, prefix):
        s3_client = boto3.client('s3')
        paginator = s3_client.get_paginator('list_objects_v2')
        folders = []
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                if key.endswith('.csv'):
                    file_name = key.split('/')[-1]
                    folders.append(file_name)
        return folders

    s3 = boto3.client('s3')
    def read_data(folder_name, which_lead, bucket='your_s3', file_prefix='Chapman_Shaoxing/ECGDataDenoised/', num_leads=12):
        object_key = file_prefix + folder_name
        response = s3_client.get_object(Bucket=bucket, Key=object_key)
        csv_content = response['Body'].read().decode('utf-8')
        df = pd.read_csv(io.StringIO(csv_content), header=None)

        column_names = [f'lead{i+1}' for i in range(num_leads)]
        df.columns = column_names

        ecg_signal = df[which_lead].values

        return ecg_signal

    folder_names = list_all_folders(bucket_name, prefix)
    print("CSV Files:", len(folder_names))
    sampling_rate = 500

    folder_names[0]

    folder_name = folder_names[0]
    which_lead = 'lead2'
    ecg_signal = read_data(folder_name, which_lead)
    print("ECG Signal:", ecg_signal)
    print("ECG Signal:", len(ecg_signal))

### MIMIC

In [9]:
if ptb_cs == 'mimic':
    s3 = boto3.client('s3')

    bucket_name = 'your_s3'
    prefix = 'mimic-iv/files_unzip/'

    def list_all_files(bucket, prefix):
        files = []
        paginator = s3.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            files.extend([content['Key'] for content in page.get('Contents', [])])
        return files

    all_files = list_all_files(bucket_name, prefix)

    print(len(all_files))
    all_files[22150:22156]

    filtered_file_names= [file for file in all_files if file.endswith('.dat') or file.endswith('.hea')]

    print('len all files', len(all_files))
    print('len filtered files', len(filtered_file_names))

    s3 = boto3.client('s3')

    def read_data(dat_file_name, which_lead):
        base_name = os.path.basename(dat_file_name).split('.')[0]

        hea_file_name = dat_file_name.replace('.dat', '.hea')

        temp_dat_path = f'/tmp/{base_name}.dat'
        temp_hea_path = f'/tmp/{base_name}.hea'

        response_dat = s3.get_object(Bucket='your_s3', Key=dat_file_name)
        response_hea = s3.get_object(Bucket='your_s3', Key=hea_file_name)

        with open(temp_dat_path, 'wb') as f:
            f.write(response_dat['Body'].read())
        with open(temp_hea_path, 'wb') as f:
            f.write(response_hea['Body'].read())

        try:
            record = wfdb.rdrecord(f'/tmp/{base_name}')

            ecg_data = record.p_signal
            ecg_signal = ecg_data[:, which_lead]

            return ecg_signal

        except Exception as e:
            print(f"Failed to process {dat_file_name}: {e}")
            return None

        finally:
            os.remove(temp_dat_path)
            os.remove(temp_hea_path)

    dat_file_names = [f for f in filtered_file_names if f.endswith('.dat')]

    print('len of dat files, which is number of patients', len(dat_file_names))

    file_name = dat_file_names[9001]
    print('example of a file name', file_name)

    ecg_signal = read_data(file_name, which_lead=1)
    print('ecg signal shape',ecg_signal.shape)

    p1037_files = [f for f in dat_file_names if '/p1052/' in f]
    print(p1037_files[0:3])

    folder_names = dat_file_names
    print('')
    print('folder_names size (number of ECGs)', len(folder_names))
    print('example of a folder_names',folder_names[0])

1600100
len all files 1600100
len filtered files 1600070
len of dat files, which is number of patients 800035
example of a file name mimic-iv/files_unzip/p1012/p10121059/s42864738/42864738.dat
ecg signal shape (5000,)
['mimic-iv/files_unzip/p1052/p10520110/s44254235/44254235.dat', 'mimic-iv/files_unzip/p1052/p10520110/s46379924/46379924.dat', 'mimic-iv/files_unzip/p1052/p10520179/s43961388/43961388.dat']

folder_names size (number of ECGs) 800035
example of a folder_names mimic-iv/files_unzip/p1000/p10000032/s40689238/40689238.dat
folder_names size (number of ECGs) 800035
example of a folder_names mimic-iv/files_unzip/p1000/p10000032/s40689238/40689238.dat


## Functions to clean ECG, delineate ECG to waves, save to pickle files

In [10]:
sampling_rate=500

In [11]:
def filter_indices(onset_offset_dict):
    arrays = np.array([
        onset_offset_dict['ECG_P_Onsets'],
        onset_offset_dict['ECG_P_Offsets'],
        onset_offset_dict['ECG_R_Onsets'],
        onset_offset_dict['ECG_R_Offsets'],
        onset_offset_dict['ECG_T_Onsets'],
        onset_offset_dict['ECG_T_Offsets']
    ])

    valid_indices = np.where(~np.isnan(arrays).any(axis=0))[0]




    p_wave_indices = [
        [onset_offset_dict['ECG_P_Onsets'][i], onset_offset_dict['ECG_P_Offsets'][i]]
        for i in valid_indices
    ]
    qrs_indices = [
        [onset_offset_dict['ECG_R_Onsets'][i], onset_offset_dict['ECG_R_Offsets'][i]]
        for i in valid_indices
    ]
    t_wave_indices = [
        [onset_offset_dict['ECG_T_Onsets'][i], onset_offset_dict['ECG_T_Offsets'][i]]
        for i in valid_indices
    ]

    return p_wave_indices, qrs_indices, t_wave_indices

In [12]:









def align_delineation_with_rpeaks(rpeaks, delineation_points, category):

    aligned_delineation = [np.nan] * len(rpeaks)

    for delineation_point in delineation_points:
        if not np.isnan(delineation_point):

            distances = delineation_point - rpeaks

            if category == "precede":
                relevant_distances = np.where(distances < 0, np.abs(distances), np.inf)
            elif category == "follow":
                relevant_distances = np.where(distances > 0, distances, np.inf)
            else:
                raise ValueError("Invalid category. Use 'precede' or 'follow'.")

            closest_idx = np.argmin(relevant_distances)

            if relevant_distances[closest_idx] != np.inf:
                aligned_delineation[closest_idx] = delineation_point


    return aligned_delineation

In [13]:
def prepare_wave_indices_for_plotting(waves_dwt):

    p_wave_indices = [
        (waves_dwt['ECG_P_Onsets'][i], waves_dwt['ECG_P_Offsets'][i])
        for i in range(len(waves_dwt['ECG_P_Onsets']))
    ]

    qrs_indices = [
        (waves_dwt['ECG_R_Onsets'][i], waves_dwt['ECG_R_Offsets'][i])
        for i in range(len(waves_dwt['ECG_R_Onsets']))
    ]

    t_wave_indices = [
        (waves_dwt['ECG_T_Onsets'][i], waves_dwt['ECG_T_Offsets'][i])
        for i in range(len(waves_dwt['ECG_T_Onsets']))
    ]

    return p_wave_indices, qrs_indices, t_wave_indices

In [14]:
def visualize_ecg_signal_with_rpeaks(ecg_signal, rpeaks):
    plt.figure(figsize=(10, 5))
    plt.plot(ecg_signal, label='ECG Signal')
    plt.plot(rpeaks, ecg_signal[rpeaks], 'ro', label='R-peaks')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.title('ECG Signal with R-peaks')
    plt.legend()
    plt.show()

def visualize_ecg_with_waves(ecg_signal, p_wave_indices, qrs_indices, t_wave_indices):
    plt.figure(figsize=(10, 5))
    plt.plot(ecg_signal, label='ECG Signal')

    for start, end in p_wave_indices:
        plt.axvspan(start, end, color='red', alpha=0.3, label='P Wave' if start == p_wave_indices[0][0] else "")
    for start, end in qrs_indices:
        plt.axvspan(start, end, color='green', alpha=0.3, label='QRS Complex' if start == qrs_indices[0][0] else "")
    for start, end in t_wave_indices:
        plt.axvspan(start, end, color='blue', alpha=0.3, label='T Wave' if start == t_wave_indices[0][0] else "")

    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.title('ECG Signal with P, QRS, and T Waves')
    plt.legend()
    plt.show()

### Process ecg signal function

In [15]:
def process_ecg_signal(ecg_signal, sampling_rate):
    """
    Process the ECG signal to extract P, QRS, and T wave data.
    """

    ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate)

    try:
        _, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)

        rpeaks = rpeaks['ECG_R_Peaks']
        rpeaks = rpeaks[~np.isnan(rpeaks)].astype(int)

        if len(rpeaks) < 4:
            return [], [], [], [], [], []

    except (ValueError, IndexError) as e:
        print(f"R-peak detection error: {e}")
        return [], [], [], [], [], []

    try:
        _, waves_dwt = nk.ecg_delineate(ecg_cleaned, rpeaks, sampling_rate=sampling_rate, method="dwt")
    except ValueError as e:
        print(f"Delineation error: {e}")
        return [], [], [], [], [], []

    if visualize == 'yes':
        print('R peaks')
        visualize_ecg_signal_with_rpeaks(ecg_signal, rpeaks)

    waves_dwt_org = waves_dwt.copy()

    p_wave_indices_bad, qrs_indices_bad, t_wave_indices_bad = prepare_wave_indices_for_plotting(waves_dwt)
    if visualize == 'yes':
        print('bad indices')
        visualize_ecg_with_waves(ecg_cleaned, p_wave_indices_bad, qrs_indices_bad,t_wave_indices_bad)
        print('p_wave_indices_bad',p_wave_indices_bad )



    category_mapping = {
    'ECG_P_Peaks': 'precede',
    'ECG_P_Onsets': 'precede',
    'ECG_P_Offsets': 'precede',
    'ECG_Q_Peaks': 'precede',
    'ECG_R_Onsets': 'precede',
    'ECG_R_Offsets': 'follow',
    'ECG_S_Peaks': 'follow',
    'ECG_T_Peaks': 'follow',
    'ECG_T_Onsets': 'follow',
    'ECG_T_Offsets': 'follow'
    }

    aligned_delineation_dict = {}
    for key, values in waves_dwt.items():
        category = category_mapping.get(key)

        if category:
            aligned_delineation = align_delineation_with_rpeaks(rpeaks, values, category=category)

            waves_dwt[key] = aligned_delineation

    p_wave_indices_good, qrs_indices_good, t_wave_indices_good = prepare_wave_indices_for_plotting(waves_dwt)
    if visualize == 'yes':
        print('good indices - after alignment function')
        visualize_ecg_with_waves(ecg_cleaned, p_wave_indices_good, qrs_indices_good,t_wave_indices_good)
        print('p_wave_indices_good after alingment',p_wave_indices_good )

    p_wave_indices, qrs_indices, t_wave_indices = filter_indices(waves_dwt)
    if visualize == 'yes':
        print('final indices - after filtering')
        visualize_ecg_with_waves(ecg_cleaned, p_wave_indices, qrs_indices,t_wave_indices)
        print('p_wave_indices before normalization',p_wave_indices)


    if visualize == 'yes':
        print('normalized signal and indices')
        visualize_ecg_with_waves(ecg_cleaned, p_wave_indices, qrs_indices,t_wave_indices)
        print('p_wave_indices after normalization',p_wave_indices)

    min_length = min(len(p_wave_indices), len(qrs_indices), len(t_wave_indices))

    filtered_p_wave_indices = []
    filtered_qrs_indices = []
    filtered_t_wave_indices = []

    for i in range(min_length):
        p_start, p_end = p_wave_indices[i]
        p_start, p_end = int(p_start), int(p_end)
        qrs_start, qrs_end = qrs_indices[i]
        qrs_start, qrs_end = int(qrs_start), int(qrs_end)
        t_start, t_end = t_wave_indices[i]
        t_start, t_end = int(t_start), int(t_end)

        filtered_p_wave_indices.append((p_start, p_end))
        filtered_qrs_indices.append((qrs_start, qrs_end))
        filtered_t_wave_indices.append((t_start, t_end))


    p_wave_data = [ecg_cleaned[start:end] for start, end in filtered_p_wave_indices]
    qrs_wave_data = [ecg_cleaned[start:end] for start, end in filtered_qrs_indices]
    t_wave_data = [ecg_cleaned[start:end] for start, end in filtered_t_wave_indices]

    if visualize == 'yes':
        print('final')
        visualize_ecg_with_waves(ecg_cleaned, filtered_p_wave_indices, filtered_qrs_indices, filtered_t_wave_indices)
        print('p_wave_indices final',filtered_p_wave_indices)

    return p_wave_data, qrs_wave_data, t_wave_data, filtered_p_wave_indices, filtered_qrs_indices, filtered_t_wave_indices

### CNN functions

# (skipped!!) Main Loop - Create patient_wave_data dictionary or Visualize wave segments (visualize = 'yes')

In [17]:
folder_names_to_visualize = folder_names[20:21]
folder_names_to_visualize

['mimic-iv/files_unzip/p1000/p10000826/s40894787/40894787.dat']

In [18]:
len(folder_names)

800035

In [19]:
visualize = 'no'

if visualize == 'yes':
    folder_names_to_run = folder_names_to_visualize
    print(folder_names_to_run)
else:
    folder_names_to_run = folder_names[:]
    print(folder_names_to_run[0:2])
    print(len(folder_names_to_run))

['mimic-iv/files_unzip/p1000/p10000032/s40689238/40689238.dat', 'mimic-iv/files_unzip/p1000/p10000032/s44458630/44458630.dat']
800035


In [20]:
def has_empty_sublists(list_of_lists):
    return any(
        (isinstance(sublist, (list, np.ndarray)) and len(sublist) == 0) or sublist is None
        for sublist in list_of_lists
    )

In [ ]:
print(ptb_cs)

patient_wave_data = {}
valid_patient_count = 0
count = 0
save_index = 1

for file_index, folder_name in enumerate(folder_names_to_run, start=1):
    start = time.time()
    count = count + 1

    if visualize == 'yes':
        print(folder_name)

    if ptb_cs =='cs':
        which_lead = 'lead2'
        patient_id = f"CS_{file_index}"
        ecg_signal = read_data(folder_name, which_lead)
    if ptb_cs =='ptb':
        patient_id = f"PTB_{file_index}"
        ecg_signal = read_data(folder_name[0], folder_name[1])
    if ptb_cs =='mimic':
        patient_id = f"mimic_{file_index}"
        which_lead = 1
        ecg_signal = read_data(folder_name, which_lead)

    try:
        p_wave, qrs_wave, t_wave, p_wave_indices, qrs_indices, t_wave_indices = process_ecg_signal(ecg_signal, sampling_rate=500)

        if not (p_wave and qrs_wave and t_wave and p_wave_indices and qrs_indices and t_wave_indices):
            print(f"Skipping patient {patient_id} due to empty wave data or indices.")
            continue

        if any(len(wave) == 0 for wave in [p_wave, qrs_wave, t_wave]):
            print(f"Skipping patient {patient_id} due to len = 0")
            continue

        skip_patient = False
        for index, sublist in enumerate(p_wave):
            if isinstance(sublist, (list, np.ndarray)) and len(sublist) == 0:
                print(f"Empty sublist found at index {index}: {sublist}")
                print(f"Skipping patient {patient_id} due to empty sublist in p_wave.")
                skip_patient = True
                break

        if skip_patient:
            continue

        skip_patient = False
        for index, sublist in enumerate(qrs_wave):
            if isinstance(sublist, (list, np.ndarray)) and len(sublist) == 0:
                print(f"Empty sublist found at index {index}: {sublist}")
                print(f"Skipping patient {patient_id} due to empty sublist in qrs_wave.")
                skip_patient = True
                break

        if skip_patient:
            continue

        skip_patient = False
        for index, sublist in enumerate(t_wave):
            if isinstance(sublist, (list, np.ndarray)) and len(sublist) == 0:
                print(f"Empty sublist found at index {index}: {sublist}")
                print(f"Skipping patient {patient_id} due to empty sublist in t_wave.")
                skip_patient = True
                break

        if skip_patient:
            continue

        raw_ecg_tensor = torch.tensor(ecg_signal).unsqueeze(0).unsqueeze(0).float()

        intermediate_feature_map = cnn_model(raw_ecg_tensor)

        p_wave_features = extract_segment_features(intermediate_feature_map, p_wave_indices)
        qrs_wave_features = extract_segment_features(intermediate_feature_map, qrs_indices)
        t_wave_features = extract_segment_features(intermediate_feature_map, t_wave_indices)

        has_nan = False
        for feature_set in [p_wave_features, qrs_wave_features, t_wave_features]:
            for feature in feature_set:
                if torch.is_tensor(feature):
                    if torch.isnan(feature).any():
                        has_nan = True
                        break
                elif isinstance(feature, (list, np.ndarray)):
                    if np.isnan(np.array(feature)).any():
                        has_nan = True
                        break
            if has_nan:
                break

        if has_nan:
            print(f"Skipping patient {patient_id} due to NaN in wave features.")
            continue



        patient_wave_data[patient_id] = {
            'p_wave': p_wave,
            'qrs_wave': qrs_wave,
            't_wave': t_wave,
            'p_wave_indices': p_wave_indices,
            'qrs_indices': qrs_indices,
            't_wave_indices': t_wave_indices,
            'p_wave_features': p_wave_features,
            'qrs_wave_features': qrs_wave_features,
            't_wave_features': t_wave_features
        }
        valid_patient_count += 1

        if valid_patient_count % 100 == 0:
            print(f"Saving batch {save_index} with 100 valid patients.")
            with open(f'{suff}patient_wave_data_{save_index}.pkl', 'wb') as f:
                pickle.dump(patient_wave_data, f)
                patient_wave_data = {}
                gc.collect()
                save_index += 1

    except:
        print(type(ecg_signal))
        print(ecg_signal.shape)
        print(ecg_signal[:10])
        if isinstance(ecg_signal, (np.ndarray, pd.Series)):
            print("Contains NaN values:", np.any(pd.isna(ecg_signal)))
            print("Number of NaN values:", np.sum(pd.isna(ecg_signal)))

if len(patient_wave_data) > 0:
    print(f"Saving final batch {save_index} with {len(patient_wave_data)} patients.")
    with open(f'{suff}patient_wave_data_{save_index}.pkl', 'wb') as f:
        pickle.dump(patient_wave_data, f)
        gc.collect()

end = time.time()
print('It took', (end - start) / 60, 'minutes')
print("Total valid patients processed:", valid_patient_count)
print("ith record checked:", count)

In [ ]:
print(patient_wave_data.keys())
print(len(patient_wave_data.keys()))


## (skipped!!) If separate pickle files exist, start here !!!!!

In [ ]:
def read_pickle_file(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)

merged_patient_wave_data = {}


num_files = 7709

file_paths = [f"{suff}patient_wave_data_{i}.pkl" for i in range(7001, num_files + 1)]

for file_path in file_paths:
    print(file_path)
    patient_wave_data = read_pickle_file(file_path)
    merged_patient_wave_data.update(patient_wave_data)

patient_wave_data = merged_patient_wave_data
merged_patient_wave_data = {}
gc.collect()

print(len(patient_wave_data.keys()))

## (skipped!!) Save the patient_wave_data

In [80]:
filtered_patient_wave_data = {patient: waves for patient, waves in patient_wave_data.items() if any(waves.values())}
patient_wave_data = filtered_patient_wave_data
filtered_patient_wave_data = {}
gc.collect()

print('number of patients' , len(patient_wave_data.keys()))

total_p_waves = 0
total_qrs_waves = 0
total_t_waves = 0

for patient, waves in patient_wave_data.items():
    p_wave_count = len(waves['p_wave'])
    qrs_wave_count = len(waves['qrs_wave'])
    t_wave_count = len(waves['t_wave'])
    total_p_waves += p_wave_count
    total_qrs_waves += qrs_wave_count
    total_t_waves += t_wave_count
    if patient == 'g1_P1':
        print(f"Patient {patient} has {p_wave_count} P waves, {qrs_wave_count} QRS waves, and {t_wave_count} T waves")

print(f"Total P waves: {total_p_waves}")
print(f"Total QRS waves: {total_qrs_waves}")
print(f"Total T waves: {total_t_waves}")
print("Number of patients:", len(patient_wave_data))

number of patients 70813
Total P waves: 704781
Total QRS waves: 704781
Total T waves: 704781
Number of patients: 70813


In [81]:
print(patient_wave_data_file_name)

./pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl


In [82]:
def add_org_to_filename(file_name):
    if file_name.endswith('.pkl'):
        return file_name.replace('.pkl', '_org.pkl')
    else:
        raise ValueError("The provided file name does not have a '.pkl' extension.")

updated_file_name = add_org_to_filename(patient_wave_data_file_name)

print(f"Original file name: {patient_wave_data_file_name}")
print(f"Updated file name: {updated_file_name}")

Original file name: ./pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl
Updated file name: ./pickles/new_pickles/MIMIC/mimic_patient_wave_data_org.pkl


In [83]:
with open(updated_file_name, "wb") as f:
    pickle.dump(patient_wave_data, f)

print(updated_file_name)
print("Number of patients:", len(patient_wave_data))
print(f"Total P waves: {total_p_waves}")
print(f"Total QRS waves: {total_qrs_waves}")
print(f"Total T waves: {total_t_waves}")

./pickles/new_pickles/MIMIC/mimic_patient_wave_data_org.pkl
Number of patients: 70813
Total P waves: 704781
Total QRS waves: 704781
Total T waves: 704781


In [85]:
total_p_waves = sum(len(waves['p_wave']) for waves in patient_wave_data.values())
total_qrs_waves = sum(len(waves['qrs_wave']) for waves in patient_wave_data.values())
total_t_waves = sum(len(waves['t_wave']) for waves in patient_wave_data.values())

print(f"Total P waves: {total_p_waves}")
print(f"Total QRS waves: {total_qrs_waves}")
print(f"Total T waves: {total_t_waves}")

Total P waves: 704781
Total QRS waves: 704781
Total T waves: 704781


# HPC signal multiprocessing via multiple cpus

In [20]:
visualize = 'no'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)

Using cuda


In [22]:
"""High-performance preprocessing (multiprocessing + xresnet1d ROI)"""

import os, gc, math, multiprocessing as mp
from tqdm import tqdm
import numpy as np
import pickle
import torch


def _finite_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x

def _valid_pair(p):
    try:
        s, e = int(p[0]), int(p[1])
        return (e > s) and np.isfinite(s) and np.isfinite(e)
    except Exception:
        return False

def _pick_first_pair(idx):
    if idx is None:
        return None
    a = np.asarray(idx)
    if a.ndim == 1 and a.size == 2:
        return tuple(a.tolist()) if _valid_pair(a) else None
    if a.ndim == 2 and a.shape[1] == 2 and a.shape[0] >= 1:
        p = a[0]
        return tuple(p.tolist()) if _valid_pair(p) else None
    return None

def _non_empty_list_of_segments(lst):

    if not lst:
        return False
    for seg in lst:
        if not hasattr(seg, "__len__") or len(seg) == 0:
            return False
    return True


def cpu_worker(args):
    """
    Read raw ECG, delineate, return everything **except** CNN features.
    """
    file_index, s3_key = args
    patient_id = f"mimic_{file_index}"
    try:
        ecg_signal = read_data(s3_key, which_lead=1)
        if ecg_signal is None:
            return None
        ecg_signal = _finite_np(ecg_signal)
        if ecg_signal.ndim != 1 or len(ecg_signal) < 1000:
            return None


        p_wave, qrs_wave, t_wave, p_idx, qrs_idx, t_idx = process_ecg_signal(ecg_signal, sampling_rate)


        if not (_non_empty_list_of_segments(p_wave) and
                _non_empty_list_of_segments(qrs_wave) and
                _non_empty_list_of_segments(t_wave)):
            return None


        p_first   = _pick_first_pair(p_idx)
        qrs_first = _pick_first_pair(qrs_idx)
        t_first   = _pick_first_pair(t_idx)
        if p_first is None and qrs_first is None and t_first is None:
            return None

        return dict(
            patient_id=patient_id,
            ecg_signal=ecg_signal,
            p_wave=p_wave,
            qrs_wave=qrs_wave,
            t_wave=t_wave,
            p_idx=p_idx,
            qrs_idx=qrs_idx,
            t_idx=t_idx
        )

    except Exception as e:

        print(f"[CPU worker] {s3_key} error: {e}")
        return None



folder_names_to_run = folder_names[:]
args_list = list(enumerate(folder_names_to_run, start=1))

print(f"Starting CPU delineation on {len(args_list)} ECGs using {mp.cpu_count()} cores")
with mp.Pool(processes=mp.cpu_count()) as pool:

    cpu_results = list(tqdm(pool.imap_unordered(cpu_worker, args_list, chunksize=32),
                            total=len(args_list)))

cpu_results = [r for r in cpu_results if r is not None]
print(f"Kept {len(cpu_results)} valid records after quality checks")

Starting CPU delineation on 800035 ECGs using 48 cores


100%|██████████| 800035/800035 [1:07:36<00:00, 197.23it/s]


Kept 772633 valid records after quality checks


In [ ]:
import importlib, xresnet_embed
importlib.reload(xresnet_embed)
from src.xresnet_embed import WaveEmbeddingExtractor
MODEL_PATH = "./models/fastai_xresnet1d101.pth"
extractor = WaveEmbeddingExtractor(model_path=MODEL_PATH, device="cuda", normalize=True)

patient_wave_data = {}
batch_size = 10000
with torch.no_grad():
    for i in tqdm(range(0, len(cpu_results), batch_size), desc="GPU batches"):
        batch = cpu_results[i:i+batch_size]
        for r in batch:
            ecg = r["ecg_signal"]
            ecg = ecg.detach().cpu().numpy().astype(np.float32) if torch.is_tensor(ecg) else np.asarray(ecg, np.float32)
            names = [k for k in ("p_idx","qrs_idx","t_idx") if r.get(k) is not None]
            segs_500 = [r[k] for k in names]
            embs = extractor.get_embeddings_roi_from_full(ecg, segs_500)
            if not np.isfinite(embs).all():
                continue
            feat = dict(zip(names, embs))
            patient_wave_data[r["patient_id"]] = {
                "p_wave": r["p_wave"],
                "qrs_wave": r["qrs_wave"],
                "t_wave": r["t_wave"],
                "p_wave_indices": r["p_idx"],
                "qrs_indices": r["qrs_idx"],
                "t_wave_indices": r["t_idx"],
                "p_wave_features": feat.get("p_idx"),
                "qrs_wave_features": feat.get("qrs_idx"),
                "t_wave_features": feat.get("t_idx"),
            }

del cpu_results
gc.collect()

import os, pickle

os.makedirs(suff, exist_ok=True)

patient_items = list(patient_wave_data.items())
total_patients = len(patient_items)
records_per_file = 100000

print(f"Saving {total_patients} patients with {records_per_file} records per file")

for file_idx in range(7):
    start_idx = file_idx * records_per_file
    end_idx = start_idx + records_per_file

    file_data = dict(patient_items[start_idx:end_idx])

    out_path = os.path.join(suff, f"mimic_patient_wave_data_{file_idx}.pkl")
    tmp_path = out_path + ".tmp"

    with open(tmp_path, "wb") as f:
        pickle.dump(file_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_path, out_path)

    print(f"Saved part {file_idx}: {len(file_data)} patients to {out_path}")

start_idx = 7 * records_per_file
file_data = dict(patient_items[start_idx:])

out_path = os.path.join(suff, f"mimic_patient_wave_data_7.pkl")
tmp_path = out_path + ".tmp"

with open(tmp_path, "wb") as f:
    pickle.dump(file_data, f, protocol=pickle.HIGHEST_PROTOCOL)
os.replace(tmp_path, out_path)

print(f"Saved part 7: {len(file_data)} patients to {out_path}")
print(f"Successfully saved data across 8 files")

[load_state_dict] missing=['8.2.0.weight', '8.2.0.bias', '8.2.0.running_mean', '8.2.0.running_var', '8.2.2.weight', '8.2.2.bias'] … unexpected=['8.4.weight', '8.4.bias', '8.6.weight', '8.6.bias', '8.6.running_mean', '8.6.running_var', '8.6.num_batches_tracked', '8.8.weight'] …
Device: cuda | C=256 | emb_dim=512 | eff_stride@100Hz=32


GPU batches: 100%|██████████| 78/78 [2:35:08<00:00, 119.34s/it]  


Saving 772633 patients with 100000 records per file
Saved part 0: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_0.pkl
Saved part 1: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_1.pkl
Saved part 2: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_2.pkl
Saved part 3: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_3.pkl
Saved part 4: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_4.pkl
Saved part 5: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_5.pkl
Saved part 6: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_6.pkl
Saved part 7: 72633 patients to ./pickles/MIMIC/mimic_patient_wave_data_7.pkl
Successfully saved data across 8 files


# Normalize it before patient_wave_data combining with the other datasets

In [ ]:


combined_p_wave_data = []
combined_qrs_wave_data = []
combined_t_wave_data = []

patient_indices = {}

for patient, waves in patient_wave_data.items():
    start_p = len(combined_p_wave_data)
    start_qrs = len(combined_qrs_wave_data)
    start_t = len(combined_t_wave_data)

    combined_p_wave_data.extend(waves['p_wave'])
    combined_qrs_wave_data.extend(waves['qrs_wave'])
    combined_t_wave_data.extend(waves['t_wave'])

    end_p = len(combined_p_wave_data)
    end_qrs = len(combined_qrs_wave_data)
    end_t = len(combined_t_wave_data)

    patient_indices[patient] = {
        'p_wave': (start_p, end_p),
        'qrs_wave': (start_qrs, end_qrs),
        't_wave': (start_t, end_t)
    }

print("Number of combined P wave entries:", len(combined_p_wave_data))
print("Number of combined QRS wave entries:", len(combined_qrs_wave_data))
print("Number of combined T wave entries:", len(combined_t_wave_data))


Number of combined P wave entries: 8144568
Number of combined QRS wave entries: 8144568
Number of combined T wave entries: 8144568


In [ ]:
def analyze_wave_data(combined_wave_data):
    nan_count = 0
    none_count = 0
    empty_count = 0
    empty_sublists = []

    for index, sublist in enumerate(combined_wave_data):
        if isinstance(sublist, (list, np.ndarray)) and len(sublist) == 0:
            empty_count += 1
            empty_sublists.append((index, sublist))
        else:
            for value in sublist:
                if value is None:
                    none_count += 1
                elif isinstance(value, float) and np.isnan(value):
                    nan_count += 1

    summary = {
        "empty_count": empty_count,
        "none_count": none_count,
        "nan_count": nan_count,
    }

    return summary, empty_sublists

In [ ]:
summary, empty_sublists = analyze_wave_data(combined_p_wave_data)

print(f"Summary: {summary}")
print("Empty Sublists (index and sublist):")
for idx, sublist in empty_sublists:
    print(f"Index: {idx}, Sublist: {sublist}")

summary, empty_sublists = analyze_wave_data(combined_qrs_wave_data)

print(f"Summary: {summary}")
print("Empty Sublists (index and sublist):")
for idx, sublist in empty_sublists:
    print(f"Index: {idx}, Sublist: {sublist}")

summary, empty_sublists = analyze_wave_data(combined_t_wave_data)

print(f"Summary: {summary}")
print("Empty Sublists (index and sublist):")
for idx, sublist in empty_sublists:
    print(f"Index: {idx}, Sublist: {sublist}")

Summary: {'empty_count': 0, 'none_count': 0, 'nan_count': 0}
Empty Sublists (index and sublist):
Summary: {'empty_count': 0, 'none_count': 0, 'nan_count': 0}
Empty Sublists (index and sublist):
Summary: {'empty_count': 0, 'none_count': 0, 'nan_count': 0}
Empty Sublists (index and sublist):


In [ ]:
def check_normalization(data, name):
    means = [np.mean(segment) for segment in data]
    stds = [np.std(segment) for segment in data]

    print(f"{name}: Mean (avg across segments) = {np.mean(means):.5f}, Std (avg across segments) = {np.mean(stds):.5f}")
    print(f"{name}: Individual segment stats -> Mean (min, max): {min(means):.5f}, {max(means):.5f}; Std (min, max): {min(stds):.5f}, {max(stds):.5f}")

check_normalization(combined_p_wave_data, "P-Wave")
check_normalization(combined_qrs_wave_data, "QRS-Wave")
check_normalization(combined_t_wave_data, "T-Wave")

P-Wave: Mean (avg across segments) = 0.00972, Std (avg across segments) = 0.03217
P-Wave: Individual segment stats -> Mean (min, max): -9.03509, 8.15303; Std (min, max): 0.00003, 4.71255
QRS-Wave: Mean (avg across segments) = 0.02102, Std (avg across segments) = 0.17118
QRS-Wave: Individual segment stats -> Mean (min, max): -6.07974, 5.24938; Std (min, max): 0.00096, 8.76508
T-Wave: Mean (avg across segments) = 0.04644, Std (avg across segments) = 0.04511
T-Wave: Individual segment stats -> Mean (min, max): -6.88726, 7.09374; Std (min, max): 0.00000, 9.85214


In [ ]:
def normalize_z_score(segment):
    if np.std(segment) == 0:
        return segment
    return (segment - np.mean(segment)) / np.std(segment)

combined_p_wave_data = [normalize_z_score(segment) for segment in combined_p_wave_data]
combined_qrs_wave_data = [normalize_z_score(segment) for segment in combined_qrs_wave_data]
combined_t_wave_data = [normalize_z_score(segment) for segment in combined_t_wave_data]

print("Wave data normalized using Z-score.")

Wave data normalized using Z-score.


In [ ]:
def check_normalization(data, name):
    means = [np.mean(segment) for segment in data]
    stds = [np.std(segment) for segment in data]

    print(f"{name}: Mean (avg across segments) = {np.mean(means):.5f}, Std (avg across segments) = {np.mean(stds):.5f}")
    print(f"{name}: Individual segment stats -> Mean (min, max): {min(means):.5f}, {max(means):.5f}; Std (min, max): {min(stds):.5f}, {max(stds):.5f}")

check_normalization(combined_p_wave_data, "P-Wave")
check_normalization(combined_qrs_wave_data, "QRS-Wave")
check_normalization(combined_t_wave_data, "T-Wave")

P-Wave: Mean (avg across segments) = -0.00000, Std (avg across segments) = 1.00000
P-Wave: Individual segment stats -> Mean (min, max): -0.00000, 0.00000; Std (min, max): 1.00000, 1.00000
QRS-Wave: Mean (avg across segments) = -0.00000, Std (avg across segments) = 1.00000
QRS-Wave: Individual segment stats -> Mean (min, max): -0.00000, 0.00000; Std (min, max): 1.00000, 1.00000
T-Wave: Mean (avg across segments) = -0.00000, Std (avg across segments) = 1.00000
T-Wave: Individual segment stats -> Mean (min, max): -0.06349, 0.00000; Std (min, max): 0.00000, 1.00000


In [ ]:
for patient, indices in patient_indices.items():
    start_p, end_p = indices['p_wave']
    patient_wave_data[patient]['p_wave'] = combined_p_wave_data[start_p:end_p]

    start_qrs, end_qrs = indices['qrs_wave']
    patient_wave_data[patient]['qrs_wave'] = combined_qrs_wave_data[start_qrs:end_qrs]

    start_t, end_t = indices['t_wave']
    patient_wave_data[patient]['t_wave'] = combined_t_wave_data[start_t:end_t]

print("Patient wave data updated with normalized values.")

Patient wave data updated with normalized values.


In [ ]:
len(patient_wave_data.keys())

772633

In [ ]:
patient_items = list(patient_wave_data.items())
total_patients = len(patient_items)
records_per_file = 100000

print(f"Saving {total_patients} patients with {records_per_file} records per file")

for file_idx in range(7):
    start_idx = file_idx * records_per_file
    end_idx = start_idx + records_per_file

    file_data = dict(patient_items[start_idx:end_idx])

    out_path = os.path.join(suff, f"mimic_patient_wave_data_normalized_{file_idx}.pkl")
    tmp_path = out_path + ".tmp"

    with open(tmp_path, "wb") as f:
        pickle.dump(file_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_path, out_path)

    print(f"Saved part {file_idx}: {len(file_data)} patients to {out_path}")

start_idx = 7 * records_per_file
file_data = dict(patient_items[start_idx:])

out_path = os.path.join(suff, f"mimic_patient_wave_data_normalized_7.pkl")
tmp_path = out_path + ".tmp"

with open(tmp_path, "wb") as f:
    pickle.dump(file_data, f, protocol=pickle.HIGHEST_PROTOCOL)
os.replace(tmp_path, out_path)

print(f"Saved part 7: {len(file_data)} patients to {out_path}")
print(f"Successfully saved data across 8 files")

print("Number of patients:", len(patient_wave_data))
print(f"Total P waves: {len(combined_p_wave_data)}")
print(f"Total QRS waves: {len(combined_qrs_wave_data)}")
print(f"Total T waves: {len(combined_t_wave_data)}")

Saving 772633 patients with 100000 records per file
Saved part 0: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_0.pkl
Saved part 1: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_1.pkl
Saved part 2: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_2.pkl
Saved part 3: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_3.pkl
Saved part 4: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_4.pkl
Saved part 5: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_5.pkl
Saved part 6: 100000 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_6.pkl
Saved part 7: 72633 patients to ./pickles/MIMIC/mimic_patient_wave_data_normalized_7.pkl
Successfully saved data across 8 files


NameError: name 'patient_wave_data_file_name' is not defined

# (skipped!!!) Check if normalized and originals are same, except the normalization

In [27]:
print(updated_file_name)
with open(updated_file_name, 'rb') as file:
    patient_wave_data_org = pickle.load(file)

len(patient_wave_data_org.keys())

./pickles/new_pickles/MIMIC/mimic_patient_wave_data_org.pkl


100000

In [28]:
import random
def get_random_keys(dictionary, n):
    return random.sample(list(dictionary.keys()), n)

random_keys = get_random_keys(patient_wave_data_org, 1)
print("Random keys:", random_keys)

Random keys: ['mimic_85205']


In [29]:
for patient in random_keys:
    print(f"Comparing lengths for patient: {patient}")

    for wave_key in [
        'p_wave', 'qrs_wave', 't_wave',
        'p_wave_indices', 'qrs_indices', 't_wave_indices',
        'p_wave_features', 'qrs_wave_features', 't_wave_features'
    ]:
        org_length = len(patient_wave_data_org[patient][wave_key])
        norm_length = len(patient_wave_data[patient][wave_key])

        if org_length != norm_length:
            print(f"  Mismatch in '{wave_key}': Original={org_length}, Normalized={norm_length}")
        else:
            print(f"  Lengths match for '{wave_key}': {org_length}")


    print("-" * 40)

    print('p_wave_indices',patient_wave_data_org[patient]['p_wave_indices'])
    print('p_wave_indices',patient_wave_data[patient]['p_wave_indices'])
    print('qrs_indices',patient_wave_data_org[patient]['qrs_indices'])
    print('qrs_indices',patient_wave_data[patient]['qrs_indices'])
    print('t_wave_indices',patient_wave_data_org[patient]['t_wave_indices'])
    print('t_wave_indices',patient_wave_data[patient]['t_wave_indices'])

    print('p_wave',patient_wave_data_org[patient]['p_wave'][0][0:5])
    print('p_wave',patient_wave_data[patient]['p_wave'][0][0:5])

    print("-" * 40)
    print('p_wave_features',patient_wave_data_org[patient]['p_wave_features'][0][0:5])
    print('p_wave_features',patient_wave_data[patient]['p_wave_features'][0][0:5])

    print("-" * 40)

Comparing lengths for patient: mimic_85205
  Lengths match for 'p_wave': 7
  Lengths match for 'qrs_wave': 7
  Lengths match for 't_wave': 7
  Lengths match for 'p_wave_indices': 7
  Lengths match for 'qrs_indices': 7
  Lengths match for 't_wave_indices': 7
  Lengths match for 'p_wave_features': 7
  Lengths match for 'qrs_wave_features': 7
  Lengths match for 't_wave_features': 7
----------------------------------------
p_wave_indices [(734, 766), (1366, 1402), (1897, 1947), (2466, 2502), (2998, 3056), (3582, 3635), (4216, 4252)]
p_wave_indices [(734, 766), (1366, 1402), (1897, 1947), (2466, 2502), (2998, 3056), (3582, 3635), (4216, 4252)]
qrs_indices [(799, 933), (1381, 1511), (2000, 2093), (2479, 2612), (3065, 3205), (3649, 3783), (4232, 4360)]
qrs_indices [(799, 933), (1381, 1511), (2000, 2093), (2479, 2612), (3065, 3205), (3649, 3783), (4232, 4360)]
t_wave_indices [(1040, 1112), (1590, 1685), (2178, 2273), (2662, 2773), (3297, 3376), (3890, 3953), (4350, 4391)]
t_wave_indices [(104

In [30]:
mismatch_found = False

for patient, indices in patient_indices.items():
    start_p, end_p = indices['p_wave']
    start_qrs, end_qrs = indices['qrs_wave']
    start_t, end_t = indices['t_wave']

    original_p_length = len(patient_wave_data_org[patient]['p_wave'])
    normalized_p_length = end_p - start_p
    if original_p_length != normalized_p_length:
        print(f"Length mismatch for patient {patient} in P waves: Original={original_p_length}, Normalized={normalized_p_length}")
        mismatch_found = True

    original_qrs_length = len(patient_wave_data_org[patient]['qrs_wave'])
    normalized_qrs_length = end_qrs - start_qrs
    if original_qrs_length != normalized_qrs_length:
        print(f"Length mismatch for patient {patient} in QRS waves: Original={original_qrs_length}, Normalized={normalized_qrs_length}")
        mismatch_found = True

    original_t_length = len(patient_wave_data_org[patient]['t_wave'])
    normalized_t_length = end_t - start_t
    if original_t_length != normalized_t_length:
        print(f"Length mismatch for patient {patient} in T waves: Original={original_t_length}, Normalized={normalized_t_length}")
        mismatch_found = True

if not mismatch_found:
    print("All lengths match between original and normalized data.")
else:
    print("Length mismatches found. Please check the above output.")

All lengths match between original and normalized data.


## Check -- be cautious about the dataset selected

In [31]:
patient_wave_data_file_name='./pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl'

total_p_waves = 0
total_qrs_waves = 0
total_t_waves = 0

with open(patient_wave_data_file_name, 'rb') as f:
    patient_wave_data = pickle.load(f)

print(patient_wave_data_file_name)
print("Number of patients:", len(patient_wave_data))

for patient, waves in patient_wave_data.items():
    p_wave_count = len(waves.get('p_wave', []))
    qrs_wave_count = len(waves.get('qrs_wave', []))
    t_wave_count = len(waves.get('t_wave', []))
    total_p_waves += p_wave_count
    total_qrs_waves += qrs_wave_count
    total_t_waves += t_wave_count
    if patient == 'g1_P1':
        print(f"Patient {patient} has {p_wave_count} P waves, {qrs_wave_count} QRS waves, and {t_wave_count} T waves")

print(f"Total P waves: {total_p_waves}")
print(f"Total QRS waves: {total_qrs_waves}")
print(f"Total T waves: {total_t_waves}")

./pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl
Number of patients: 100000
Total P waves: 995401
Total QRS waves: 995401
Total T waves: 995401


In [32]:
check_file_name = './pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl'

print(check_file_name)
with open(check_file_name, 'rb') as file:
    patient_wave_data_check = pickle.load(file)

len(patient_wave_data_check.keys())

./pickles/new_pickles/MIMIC/mimic_patient_wave_data.pkl


100000

In [33]:
for patient, waves in patient_wave_data_check.items():
    for wave_type, wave_list in waves.items():
        if any(len(wave) == 0 for wave in wave_list):
            print(f"Empty sublist found for patient: {patient}, wave type: {wave_type}")

In [34]:
for patient, waves in patient_wave_data.items():
    lengths = {wave_type: len(wave_list) for wave_type, wave_list in waves.items()}
    if len(set(lengths.values())) > 1:
        print(f"Wave length mismatch for patient {patient}: {lengths}")